In [ ]:
!git clone https://github.com/aresu-1704/ultralytics_custom.git #Fork custom ultralytics
%cd ultralytics_custom
!pip install -q -e .

!pip install -q ultralytics

from ultralytics import YOLO # And RTDETR if you train RT-DETR
import multiprocessing
from glob import glob
import os
import random
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import yaml
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

%cd ..

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("aresusayhi/vr-tsd-vietnam-traffic-signs")
print("Đường dẫn dataset:", path)

In [ ]:
%%writefile /content/VSYolo.yaml
nc: 58
end2end: True
reg_max: 1
scales:
  n: [0.5, 0.25, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]

  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]

  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]

  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]

  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, CBAM, [256]] # 10

# YOLOv10 head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]] # 16

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]
  - [-1, 2, CBAM, [32]] # 20

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 2, CBAM, [64]] # 24

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 2, CBAM, [128]] # 28

  - [[20, 24, 28], 1, Detect, [nc]]

In [ ]:
def main():
    model = YOLO("/content/VSYolo.yaml") # Or use yolo26n.yaml, yolo12n.yaml, if you want to train rt-detr-l.yaml, change YOLO to RTDETR
    model.train(
        data='', # Dataset path yaml
        epochs=300,
        imgsz=640,
        batch=64,
        pretrained=False,
        device="cuda",
        verbose=True,
        plots=True,
        resume=False # True if resume from last checkpoint
    )

if __name__ == '__main__':
    multiprocessing.freeze_support()
    main()

In [ ]:
model = YOLO("") #Path of your best.pt
model.to('cuda')

data_yaml = "" # Dataset path yaml

if __name__ == '__main__':
    metrics = model.val(data=data_yaml, split='test', save_json=True, end2end=False) # Disable end2end if you want highest performance

In [ ]:
!yolo export model='Path of your best.pt' format=onnx imgsz=640 simplify=True